# V2 Phase 11 — Colab GPU live artefact (`llama_cpp`)

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

Validates the same live runner used by `app/streamlit_app.py` on Colab T4:

- Known-good frozen FinQA question `finqa_test_1000` (ROI, not ending value)
- Additional numerical frozen question `finqa_test_1012` (percentage change)
- Insufficient-evidence live question (SpaceX FY2025 — not in the frozen 140/40)
- All three architectures independently
- UQ display: calculated confidence, **locked T=0.65** from `threshold.lock.json`, never show 0 for a missing score

Uses **llama_cpp + Qwen3-8B**, not mock. Does **not** run the 140-question benchmark. Does **not** start Phase 12.

## Setup

Push latest V2 (prompts, UQ display helpers, `runtime_guard.py`) to branch `main`, then run **this notebook on Colab GPU**. Do not start Streamlit on the Mac.

Requires Phase 8 KB on Drive at `MyDrive/MSc-RAG/artifacts/knowledge_base/`.

**Outputs:** `results/config/phase11_colab_live_demo.json`, `phase11_smoke_test.json`, `phase11_live_smoke.json`

**Live demo (Phase 20):** section 5 runs `scripts/run_live_demo.py` (known-good frozen, fresh KB question, insufficient-evidence) at locked T=0.65. Section 8 starts Streamlit. Never open `127.0.0.1:8501` on the Mac. Do not run the 140-question benchmark.

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'main'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not (V2_ROOT / 'scripts' / 'run_live_demo.py').is_file():
    raise FileNotFoundError(f'Phase 20 script missing at {V2_ROOT}. Push Phase 20 to GitHub first.')
if not (V2_ROOT / 'src' / 'models' / 'runtime_guard.py').is_file():
    raise FileNotFoundError(
        f'runtime_guard.py missing at {V2_ROOT}. '
        'Push the Phase 11 Colab live-demo connection fix to GitHub, then re-run this cell.'
    )

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Restore knowledge base from Drive (or rebuild)

Does **not** copy the Mac Chroma database. Reuses the Colab-built index from Phase 8 when available.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')
restored = True

for rel in ('artifacts/knowledge_base/index', 'artifacts/knowledge_base/documents'):
    src = DRIVE_ROOT / rel
    dst = V2 / 'knowledge_base' / rel.split('/')[-1]
    if not src.is_dir():
        print('Missing on Drive:', src)
        restored = False
        break
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('restored', dst)

if not restored:
    print('Drive KB not found — falling back to build_index.py (Option B)...')
    !PYTHONPATH=. python scripts/build_index.py --distractors 50

## 4. Index preflight validation

In [ ]:
!PYTHONPATH=. python scripts/validate_kb_index.py

## 5. Known-good + insufficient-evidence (`llama_cpp`, all three architectures)

Runs the same `run_live_comparison()` used by Streamlit:

1. Frozen known-good question `finqa_test_1000` (ROI, not ending investment value)
2. Additional numerical frozen question `finqa_test_1012` (percentage change)
3. Insufficient-evidence live question (SpaceX FY2025 / Starship — not in the frozen 140/40)

Prints the **displayed** UQ confidence and **locked T=0.65**. Abstention uses the official lock (not smoke 0.55).

This cell must run **on Colab GPU**. It fails if the device would be `mps_capable_host` or if CUDA is missing. It does **not** start the 140-question benchmark.

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys
from datetime import datetime, timezone

V2 = Path('/content/capstone-rag/V2')
if platform.system() == 'Darwin' or not Path('/content').exists():
    raise RuntimeError(
        'This cell is not on Google Colab. '
        'backend=llama_cpp + device=mps_capable_host + 127.0.0.1:8501 means the Mac Streamlit process. '
        'Open this notebook on Colab with GPU (T4).'
    )

os.chdir(V2)
sys.path.insert(0, str(V2))
os.environ['V2_LIVE_BACKEND'] = 'llama_cpp'
os.environ['V2_FORBID_MOCK'] = '1'
os.environ['V2_REQUIRE_CUDA'] = '1'

from src.models.runtime_guard import verify_live_llama_cpp_runtime
from src.rag.live import (
    FRESH_KB_QUESTION,
    INSUFFICIENT_EVIDENCE_QUESTION,
    INSUFFICIENT_EVIDENCE_QUESTION_ID,
    KNOWN_GOOD_QUESTION_ID,
    LIVE_ARCHITECTURES,
    format_confidence_display,
    format_threshold_display,
    load_frozen_questions,
    run_live_comparison,
)
from src.rag.schema import ARCHITECTURE_MULTI_AGENT_UQ, RAGCaseResult
from src.utils import create_run_id

runtime = verify_live_llama_cpp_runtime(require_cuda=True)
print('RUNTIME LOCK:', json.dumps(runtime, indent=2))
if runtime['backend'] != 'llama_cpp' or runtime['device'] != 'cuda' or not runtime.get('gpu'):
    raise RuntimeError(f'Refusing FinQA live run: {runtime}')

frozen = load_frozen_questions()
row = next((item for item in frozen if item['id'] == KNOWN_GOOD_QUESTION_ID), None)
if row is None:
    raise RuntimeError(f'Frozen question {KNOWN_GOOD_QUESTION_ID} not found (CSV was not modified; loader failed).')

jobs = [
    {
        'question': row['question'],
        'question_id': row['id'],
        'question_source': 'frozen',
        'reference_answer': row.get('program_answer'),
    },
    {
        'question': extra['question'],
        'question_id': extra['id'],
        'question_source': 'frozen',
        'reference_answer': extra.get('program_answer'),
    },
    {
        'question': INSUFFICIENT_EVIDENCE_QUESTION,
        'question_id': INSUFFICIENT_EVIDENCE_QUESTION_ID,
        'question_source': 'insufficient',
        'reference_answer': None,
    },
]

run_id = create_run_id('phase20')
summaries = []
comparisons = []
any_error = False
for job in jobs:
    comparison = run_live_comparison(
        job['question'],
        question_id=job['question_id'],
        question_source=job['question_source'],
        reference_answer=job.get('reference_answer'),
        backend_name='llama_cpp',
        run_id=run_id,
    )
    payload = comparison.to_dict()
    comparisons.append(payload)
    fp = payload.get('fingerprint') or {}
    device = fp.get('device')
    gpu = fp.get('gpu') or {}
    print('=' * 72)
    print(job['question_source'], job['question_id'])
    print('backend:', payload.get('backend'), 'device:', device, 'gpu:', gpu)
    if payload.get('backend') != 'llama_cpp' or device == 'mps_capable_host' or not gpu.get('available'):
        raise RuntimeError(f'Not a Colab T4 result: backend={payload.get("backend")} device={device} gpu={gpu}')
    if payload.get('error'):
        any_error = True
    for architecture in LIVE_ARCHITECTURES:
        case = (payload.get('results') or {}).get(architecture) or {}
        result = RAGCaseResult(**case)
        disp_conf = format_confidence_display(result)
        disp_thr = format_threshold_display(result)
        uq = (case.get('configuration') or {}).get('uncertainty_result') or {}
        print(
            architecture,
            'decision=', case.get('decision'),
            'displayed_confidence=', disp_conf,
            'displayed_threshold=', disp_thr,
            'raw_uq_confidence=', uq.get('confidence'),
            'n_evidence=', len(case.get('retrieved_evidence') or []),
            'device=', case.get('device'),
            'gpu=', case.get('gpu'),
            'error=', case.get('error'),
        )
        if case.get('device') == 'mps_capable_host' or not case.get('gpu'):
            raise RuntimeError(f'{architecture} reported Mac/empty GPU: {case.get("device")} {case.get("gpu")}')
        if architecture == ARCHITECTURE_MULTI_AGENT_UQ:
            raw = uq.get('confidence')
            if raw is not None and float(raw) > 0 and disp_conf in {'0', '0.0', '0.0000'}:
                raise RuntimeError('UQ display would show 0 while raw confidence is non-zero. Display fix missing.')
            if disp_conf == 'n/a' and raw is None:
                print('UQ confidence n/a (not a fabricated 0.0)')
        if case.get('error'):
            any_error = True
        summaries.append(
            {
                'question_source': job['question_source'],
                'question_id': job['question_id'],
                'architecture': architecture,
                'decision': case.get('decision'),
                'displayed_confidence': disp_conf,
                'displayed_threshold': disp_thr,
                'raw_uq_confidence': uq.get('confidence'),
                'n_evidence': len(case.get('retrieved_evidence') or []),
            }
        )

record = {
    'phase': 11,
    'test_name': 'phase11_colab_live_demo_known_good_and_insufficient',
    'question_ids': [KNOWN_GOOD_QUESTION_ID, 'fresh', INSUFFICIENT_EVIDENCE_QUESTION_ID],
    'backend': 'llama_cpp',
    'device': device,
    'gpu': gpu,
    'runtime_lock': runtime,
    'run_id': run_id,
    'summaries': summaries,
    'comparisons': comparisons,
    'status': 'FAIL' if any_error else 'PASS',
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
}
out = V2 / 'results' / 'config' / 'phase11_colab_live_demo.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(record, indent=2) + '\n', encoding='utf-8')
print('Wrote', out)
phase20 = {
    'phase': 20,
    'test_name': 'phase20_live_artefact',
    'backend': 'llama_cpp',
    'official_colab_t4_llama_cpp': True,
    'locked_threshold': 0.65,
    'used_precomputed_benchmark_lookup': False,
    'used_rag_rerun_of_420': False,
    'device': device,
    'gpu': gpu,
    'runtime_lock': runtime,
    'run_id': run_id,
    'summaries': summaries,
    'status': record['status'],
    'recorded_at_utc': record['recorded_at_utc'],
}
p20 = V2 / 'results' / 'config' / 'phase20_live_demo_summary.json'
p20.write_text(json.dumps(phase20, indent=2) + '\n', encoding='utf-8')
print('Wrote', p20)
print('status:', record['status'])
print(json.dumps(summaries, indent=2))

## 6. Check UI-equivalent fields

Confirms both live questions, displayed UQ confidence (not 0), and locked T=0.65 labels.

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase11_runtime_fingerprint.json')
smoke = Path('results/config/phase11_smoke_test.json')
detail = Path('results/config/phase11_live_smoke.json')
live = Path('results/config/phase11_colab_live_demo.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
print('detail:', detail.is_file())
print('colab_live_demo:', live.is_file())
if live.is_file():
    live_data = json.loads(live.read_text())
    print('live status:', live_data.get('status'))
    print('live backend:', live_data.get('backend'))
    print('live device:', live_data.get('device'))
    print('live gpu:', live_data.get('gpu'))
    print('live question_ids:', live_data.get('question_ids') or live_data.get('question_id'))
    if live_data.get('device') == 'mps_capable_host':
        raise RuntimeError('phase11_colab_live_demo.json is a Mac result, not Colab T4.')
    for row in live_data.get('summaries') or []:
        print(
            row.get('question_source'),
            row.get('architecture'),
            'decision=', row.get('decision'),
            'displayed_confidence=', row.get('displayed_confidence'),
            'displayed_threshold=', row.get('displayed_threshold'),
            'raw_uq=', row.get('raw_uq_confidence'),
        )
        if row.get('architecture') == 'multi_agent_uq' and row.get('displayed_confidence') in {'0', '0.0', '0.0000'}:
            raw = row.get('raw_uq_confidence')
            if raw not in (None, 0, 0.0):
                raise RuntimeError('UQ UI would show 0 while raw confidence is non-zero.')
        if row.get('architecture') == 'multi_agent_uq' and 'locked' not in str(row.get('displayed_threshold') or '').lower():
            raise RuntimeError('UQ threshold must display locked T=0.65, not smoke 0.55.')
p20 = Path('results/config/phase20_live_demo_summary.json')
print('phase20_summary:', p20.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', data.get('actual'))
    env = data.get('environment') or {}
    print('device:', env.get('device'))
    print('gpu:', (env.get('gpu') or {}).get('name'))
    print('backend:', (env.get('model_config') or {}).get('backend'))
if detail.is_file():
    detail_data = json.loads(detail.read_text())
    print('backend:', detail_data.get('backend'))
    for comparison in detail_data.get('comparisons', []):
        print('---')
        print('source:', comparison.get('question_source'), 'qid:', comparison.get('question_id'))
        print('question:', (comparison.get('question') or '')[:160])
        for architecture, case in (comparison.get('results') or {}).items():
            vr = case.get('verification_result') or {}
            print(
                architecture,
                'n_evidence=', len(case.get('retrieved_evidence') or []),
                'answer_len=', len(case.get('answer') or ''),
                'verify=', vr.get('verification_score'),
                'confidence=', case.get('confidence'),
                'threshold=', case.get('threshold'),
                'decision=', case.get('decision'),
                'error=', case.get('error'),
            )

## 7. Save Phase 11 Colab results to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=True)
V2 = Path('/content/capstone-rag/V2')
dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase11')
dest.mkdir(parents=True, exist_ok=True)
for name in (
    'phase11_runtime_fingerprint.json',
    'phase11_smoke_test.json',
    'phase11_live_smoke.json',
    'phase11_colab_live_demo.json',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, dest / name)
        print('copied', name)
print('Done:', dest)

## 8. Start Streamlit inside this Colab runtime

Starts the existing `app/streamlit_app.py` **on this Colab VM** and exposes it with Colab's built-in `proxyPort` (no extra tunnel binary).

- Do **not** open `http://127.0.0.1:8501` on your Mac. That is the local Streamlit process (`mps_capable_host`).
- Open **only** the printed Colab proxy URL, or use the iframe in the next cell.
- Backend is locked to `llama_cpp` / Qwen3-8B. Mock and Ollama are forbidden.
- Streamlit question sources: Fresh, Frozen `finqa_test_1000`, and **Insufficient-evidence demo**.
- UQ cards must show the calculated confidence and **T=0.65 (locked)** — not smoke 0.55.
- Do **not** start the 140-question benchmark.

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys

V2 = Path('/content/capstone-rag/V2')
if platform.system() == 'Darwin' or not Path('/content').exists():
    raise RuntimeError(
        'This cell is not running on Google Colab. '
        'backend=mock and device=mps_capable_host mean Streamlit was started on the Mac. '
        'Open this notebook on Colab with GPU (T4) and run all cells there.'
    )

os.chdir(V2)
sys.path.insert(0, str(V2))
os.environ['V2_LIVE_BACKEND'] = 'llama_cpp'
os.environ['V2_FORBID_MOCK'] = '1'
os.environ['V2_REQUIRE_CUDA'] = '1'

from src.models.runtime_guard import verify_live_llama_cpp_runtime

runtime = verify_live_llama_cpp_runtime(require_cuda=True)
print(json.dumps(runtime, indent=2))
if runtime['backend'] != 'llama_cpp' or runtime['device'] != 'cuda':
    raise RuntimeError(f'Refusing to start Streamlit: {runtime}')
print('RUNTIME LOCKED: llama_cpp on', runtime.get('gpu'))

In [ ]:
from pathlib import Path
import platform

if platform.system() == 'Darwin' or not Path('/content').exists():
    raise RuntimeError('Not on Colab. Do not start a tunnel from the Mac.')

from google.colab.output import eval_js  # noqa: F401

print('Using Colab kernel.proxyPort — no extra tunnel binary.')
print('Do not open http://127.0.0.1:8501 on your Mac.')

In [ ]:
import json
import os
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from google.colab import output
from google.colab.output import eval_js

V2 = Path('/content/capstone-rag/V2')
if not Path('/content').exists():
    raise RuntimeError('Not on Colab. Do not start Streamlit on the Mac for this demo.')
os.chdir(V2)

def _wait_port(port: int, timeout: int = 90) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        sock = socket.socket()
        try:
            sock.connect(('127.0.0.1', port))
            return True
        except OSError:
            time.sleep(1)
        finally:
            sock.close()
    return False

os.system("pkill -f 'streamlit run app/streamlit_app.py' || true")
time.sleep(2)

env = os.environ.copy()
env['PYTHONPATH'] = str(V2)
env['V2_LIVE_BACKEND'] = 'llama_cpp'
env['V2_FORBID_MOCK'] = '1'
env['V2_REQUIRE_CUDA'] = '1'
env['STREAMLIT_SERVER_ENABLE_CORS'] = 'false'
env['STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION'] = 'false'

st_log = Path('/tmp/v2_streamlit.log')
st_handle = st_log.open('w')

st_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', 'app/streamlit_app.py',
        '--server.port=8501',
        '--server.address=0.0.0.0',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
        '--browser.gatherUsageStats=false',
    ],
    cwd=str(V2),
    env=env,
    stdout=st_handle,
    stderr=subprocess.STDOUT,
)
print('Streamlit pid on Colab VM:', st_proc.pid)
if not _wait_port(8501):
    print(st_log.read_text()[-3000:])
    raise RuntimeError('Streamlit did not start inside this Colab runtime.')
print('Streamlit bound on the Colab VM only. Do not open 127.0.0.1:8501 on the Mac.')
try:
    health = urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=15)
    print('Streamlit health:', health.status, health.read().decode('utf-8', errors='replace').strip())
except Exception as exc:
    print('Streamlit health check failed:', exc)
    print(st_log.read_text()[-2000:])
    raise RuntimeError('Streamlit process is up but the app is not healthy.') from exc

url = eval_js('google.colab.kernel.proxyPort(8501)')
if not url:
    raise RuntimeError('Colab proxyPort returned no URL. Re-run this cell.')

Path('/tmp/v2_live_demo_url.txt').write_text(str(url) + '\n', encoding='utf-8')
print('=' * 72)
print('COLAB LIVE URL — open this (not 127.0.0.1:8501):')
print(url)
print('=' * 72)
output.serve_kernel_port_as_iframe(8501, height=900)
print('Backend locked to llama_cpp. Mock and Ollama are forbidden.')
print('If Runtime shows mps_capable_host or gpu=null, you opened the Mac app. Close it.')
print('Section 5 already ran the Phase 20 live demo script on this GPU.')
print('In the UI: locked T=0.65; Insufficient-evidence demo should be able to ABSTAIN.')

## 9. Keep Streamlit alive

Run this cell and **leave it running** while you use the browser URL.

Interrupt this cell only when the manual live test is finished, then run section 10.

In [ ]:
import socket
import time
from pathlib import Path

url_file = Path('/tmp/v2_live_demo_url.txt')
url = url_file.read_text(encoding='utf-8').strip() if url_file.is_file() else '(URL cell not run)'
print('COLAB LIVE URL (not 127.0.0.1:8501):', url)
print('Keep this cell running. Interrupt when the live demo is done.')

while True:
    sock = socket.socket()
    try:
        sock.connect(('127.0.0.1', 8501))
        alive = True
    except OSError:
        alive = False
    finally:
        sock.close()
    if not alive:
        raise RuntimeError('Streamlit is no longer listening on :8501. Re-run section 8.')
    time.sleep(30)

## 10. After the manual browser test — save live-session record

Run this cell **only after** you entered a fresh question in the browser and ran all three architectures.

Set `MANUAL_TEST_COMPLETED = True` and fill the observed fields. Do not mark PASS unless you actually saw the UI results. This cell does **not** invent answers.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from shutil import copy2

MANUAL_TEST_COMPLETED = False  # set True only after the browser test
FRESH_QUESTION_USED = ''  # paste the fresh question you typed
OBSERVED_DECISIONS = {
    'single_agent': '',      # ANSWER | ERROR | UNAVAILABLE
    'multi_agent': '',
    'multi_agent_uq': '',    # ANSWER | ABSTAIN | ERROR | UNAVAILABLE
}
N_EVIDENCE = {
    'single_agent': None,
    'multi_agent': None,
    'multi_agent_uq': None,
}
SAW_FIELDS = {
    'retrieved_evidence': False,
    'retrieval_scores_metadata': False,
    'generated_answer': False,
    'verification': False,
    'confidence': False,
    'threshold': False,
    'decision': False,
}
NOTES = ''

if not MANUAL_TEST_COMPLETED:
    raise RuntimeError('Set MANUAL_TEST_COMPLETED = True only after you finished the browser live test.')

url_file = Path('/tmp/v2_live_demo_url.txt')
record = {
    'phase': 11,
    'test_name': 'phase11_colab_streamlit_manual_live',
    'command': 'notebooks/colab_phase11_live.ipynb section 8–10; streamlit run app/streamlit_app.py; V2_LIVE_BACKEND=llama_cpp',
    'backend': 'llama_cpp',
    'mock_used': False,
    'browser_url_file': str(url_file) if url_file.is_file() else None,
    'fresh_question_used': FRESH_QUESTION_USED,
    'observed_decisions': OBSERVED_DECISIONS,
    'n_evidence': N_EVIDENCE,
    'saw_fields': SAW_FIELDS,
    'notes': NOTES,
    'status': 'PASS' if all(SAW_FIELDS.values()) and FRESH_QUESTION_USED.strip() else 'NEEDS VERIFICATION',
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
}
out = Path('/content/capstone-rag/V2/results/config/phase11_colab_live_demo.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(record, indent=2) + '\n', encoding='utf-8')
print('Wrote', out)
print(json.dumps(record, indent=2))

drive_dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase11')
if drive_dest.parent.is_dir():
    drive_dest.mkdir(parents=True, exist_ok=True)
    copy2(out, drive_dest / out.name)
    print('Copied to', drive_dest / out.name)
else:
    print('Drive folder not mounted; local record only.')

print('Copy phase11_colab_live_demo.json into local V2/results/config/ then ask for evidence/master-record update.')